# CS781 Phase 3 - submissions from the cross-validation bundles (CPU only)
The best leaderboard score so far (**0.6352**) is `ens_top10` + the **Dissent vote rule** (a sentence becomes Dissent when
any member predicts Dissent). Dev has no Dissent, so only the leaderboard can measure Dissent. The 40 cross-validation
models (D and E, 5 folds x 4 variants) give **40 new, partly independent Dissent voters**. Each file below changes one
thing relative to the 0.6352 recipe, so its score difference says whether that change helps:

| group | question |
|---|---|
| more voters | do the 40 CV fold models find Dissent sentences the top 10 missed? (k=1 and k=2) |
| more members | does averaging the CV models into the top 10 help the other labels on test? |
| shape rules | Dissent is a contiguous block (a separate opinion). Does filling gaps / dropping isolated Dissent sentences help? |
| CV only | how do the 40 CV models do on their own with the same rule? |

Files are numbered in submission order; the first 5 are the most informative. A file identical to the 0.6352 file
(or to an earlier file here) is skipped. `vs 0.6352` = test sentences whose label differs from the 0.6352 file.

**Inputs:** `CS781-P3-Bundle` (bundle_A ... bundle_extra) + `CS781-P3-CV-Bundles` + `CS781-P3-Code`.
Internet ON, `HF_TOKEN`, **Accelerator: None**. Save & Run All (~5 min).

In [ ]:
import os, sys, json, re, glob, shutil, subprocess, zipfile
W = "/kaggle/working"
WORK, P3DATA, RUNS = f"{W}/p3code", f"{W}/p3data", f"{W}/runs"
def find_dir(name, root="/kaggle/input"):
    hits = sorted(glob.glob(os.path.join(root, "**", name), recursive=True), key=len)
    assert hits, f"{name} not found under {root}"
    return os.path.dirname(hits[0])
shutil.rmtree(WORK, ignore_errors=True)
shutil.copytree(find_dir("rr_train.py"), WORK)
os.chdir(WORK); sys.path.insert(0, WORK)
from huggingface_hub import hf_hub_download, list_repo_files
from kaggle_secrets import UserSecretsClient
REPO = "Exploration-Lab/CS-781-Capstone"
tok_ = UserSecretsClient().get_secret("HF_TOKEN")
os.makedirs(P3DATA, exist_ok=True)
for f in list_repo_files(REPO, repo_type="dataset", token=tok_):
    if f.startswith("Phase_3") and f.endswith(".json"):
        shutil.copy(hf_hub_download(REPO, f, repo_type="dataset", token=tok_), os.path.join(P3DATA, os.path.basename(f)))
import rr_data
for dom in ("CL", "IT"):
    json.dump({x["id"]: x["labels"] for x in rr_data.load_split(f"{P3DATA}/{dom}_dev.json")}, open(f"{W}/dev_truth_{dom}.json", "w"))
# ---- runs from every bundle attached (zip files, or folders Kaggle already unzipped)
NAME_OK = re.compile(r"^([A-E]\d\d_\w+|[DE]cv_\w+_f\d)$")
os.makedirs(RUNS, exist_ok=True)
for z in glob.glob("/kaggle/input/**/bundle_*.zip", recursive=True):
    zipfile.ZipFile(z).extractall(RUNS)
for p in glob.glob("/kaggle/input/**/summary.json", recursive=True):
    d = os.path.dirname(p); t = os.path.basename(d)
    if NAME_OK.match(t) and not os.path.exists(f"{RUNS}/{t}"):
        shutil.copytree(d, f"{RUNS}/{t}")
ALL = sorted(t for t in os.listdir(RUNS) if NAME_OK.match(t) and os.path.exists(f"{RUNS}/{t}/test_prob.npz"))
CV = [t for t in ALL if "cv_" in t]
OLD = [t for t in ALL if "cv_" not in t]
print(len(CV), "cross-validation runs;", len(OLD), "earlier runs (only needed for the top10 transfer test)")

## Plan (edit here)

In [ ]:
N_SUBMISSIONS = 10      # set to 5 to build only the first five
import numpy as np, csv
import rr_data as D
labels = D.load_vocab(f"{P3DATA}/label_vocab.json"); DIS = labels.index("Dissent")
def summ(t): return json.load(open(f"{RUNS}/{t}/summary.json"))
_c = {}
def probs(t, split):
    if (t, split) not in _c:
        f = np.load(f"{RUNS}/{t}/{split}_prob.npz")
        _c[(t, split)] = {str(k): f[f"p{i}"].astype(np.float32) for i, k in enumerate(f["ids"])}
    return _c[(t, split)]

ranked = sorted([t for t in OLD if summ(t)["dev"]["macro_f1"] >= 0.40], key=lambda t: -summ(t)["dev"]["macro_f1"])
TOP10, TOP20 = ranked[:10], ranked[:20]
TOP3PM = [t for m in "ABCDE" for t in [x for x in ranked if x.startswith(m)][:3]]
CV40 = sorted(CV)
CVBASE = [t for t in CV40 if "_base_" in t]
print("top10:", TOP10)
print(len(CV40), "CV fold models;", len(CVBASE), "of them base")

# (name, members (averaged), Dissent voters, k, shape rule, why)
PLAN = [
    ("top10_votersCV40_k1",   TOP10, TOP10 + CV40, 1, None,  "0.6352 recipe, + the 40 CV models as extra Dissent voters"),
    ("top10+cv40_dissent1",   TOP10 + CV40, TOP10 + CV40, 1, None, "same voters, CV models also averaged in (members)"),
    ("top10_dissent1_fill",   TOP10, TOP10, 1, "fill",  "0.6352 recipe + fill gaps (<=5 sentences) between Dissent sentences"),
    ("top10_votersCV40_k2",   TOP10, TOP10 + CV40, 2, None, "50 voters, Dissent needs >= 2 votes (fewer false Dissent)"),
    ("top10_dissent1_min3",   TOP10, TOP10, 1, "min3",  "0.6352 recipe, drop Dissent in docs with < 3 Dissent sentences"),
    ("cv40_dissent1",         CV40, CV40, 1, None,      "CV models alone with the Dissent rule (compare 0.6352)"),
    ("top20_dissent1",        TOP20, TOP20, 1, None,    "20 best earlier runs with the Dissent rule"),
    ("top10_votersCV40_k1_fill", TOP10, TOP10 + CV40, 1, "fill", "file 1 + gap filling"),
    ("top10+cvbase_dissent1", TOP10 + CVBASE, TOP10 + CVBASE, 1, None, "top10 + only the 10 CV base models"),
    ("top3permodel_votersCV40_k1", TOP3PM, TOP3PM + CV40, 1, None, "second-best ensemble (0.6135) + CV voters"),
    # backups (used only if a file above is identical to an earlier one)
    ("top10_votersCV40_k3",   TOP10, TOP10 + CV40, 3, None, "backup: >= 3 votes"),
    ("top10_votersCVbase_k1", TOP10, TOP10 + CVBASE, 1, None, "backup: only CV base models as extra voters"),
]
REFERENCE = (TOP10, TOP10, 1, None)    # the 0.6352 file (18_ens_top10_dissent1), rebuilt for comparison

## Build, check, score on dev

In [ ]:
def predict(members, voters, k, rule, split):
    out, changed = {}, 0
    M = [probs(t, split) for t in members]; V = [probs(t, split) for t in voters]
    for doc in M[0]:
        base = np.mean([m[doc] for m in M], 0).argmax(1)
        votes = np.sum([v[doc].argmax(1) == DIS for v in V], 0)
        y = np.where(votes >= k, DIS, base)
        if rule == "fill":                                      # contiguous block: fill short gaps
            idx = np.where(y == DIS)[0]
            for a, b in zip(idx[:-1], idx[1:]):
                if 1 < b - a <= 6:
                    y[a:b] = DIS
        if rule == "min3" and 0 < (y == DIS).sum() < 3:         # isolated Dissent is likely noise -> next-best label
            nod = np.mean([m[doc] for m in M], 0); nod[:, DIS] = -1
            y = np.where(y == DIS, nod.argmax(1), y)
        changed += int(((y == DIS) & (base != DIS)).sum())
        out[doc] = [labels[i] for i in y]
    return out, changed

test_docs = [D.RRDoc(x["id"], D.DOMAINS.index(dm), x["text"]) for dm in D.DOMAINS
             for x in D.load_split(D.split_path(P3DATA, dm, "test"))]
def official(pred):
    p = f"{W}/_dev_pred.json"; json.dump(pred, open(p, "w"))
    r = subprocess.run(f"python evaluate_phase3.py --truth-cl {W}/dev_truth_CL.json --truth-it {W}/dev_truth_IT.json "
                       f"--prediction {p} --labels {P3DATA}/label_vocab.json", shell=True, capture_output=True, text=True, cwd=WORK)
    m = re.search(r"Macro-F1:\s+([0-9.]+)", r.stdout)
    return float(m.group(1)) if m else float("nan")

ref, _ = predict(*REFERENCE, "test")
ref_dev, _ = predict(*REFERENCE, "dev")
print(f"reference (0.6352 file rebuilt): test Dissent={sum(v.count('Dissent') for v in ref.values())}, dev {official(ref_dev):.4f}")
SUB = f"{W}/submissions"; shutil.rmtree(SUB, ignore_errors=True); os.makedirs(SUB)
seen = {json.dumps(ref, sort_keys=True): "the 0.6352 file"}; rows = []
for name, mem, vot, k, rule, why in PLAN:
    if len(rows) == N_SUBMISSIONS:
        break
    test, _ = predict(mem, vot, k, rule, "test")
    key = json.dumps(test, sort_keys=True)
    if key in seen:
        print(f"skip {name}: identical to {seen[key]}"); continue
    seen[key] = name
    D.validate_submission(test, test_docs, labels)
    dev, dev_changed = predict(mem, vot, k, rule, "dev")
    fn = f"cvs{len(rows) + 1:02d}_{name}.zip"
    with zipfile.ZipFile(f"{SUB}/{fn}", "w", zipfile.ZIP_DEFLATED) as z:
        z.writestr("prediction.json", json.dumps(test, ensure_ascii=False))
    rows.append(dict(file=fn, dev_macro_f1=round(official(dev), 4), members=len(mem), voters=len(vot), k=k, rule=rule or "",
                     test_dissent=sum(v.count("Dissent") for v in test.values()),
                     test_docs_with_dissent=sum("Dissent" in v for v in test.values()),
                     vs_best=sum(a != b for d in test for a, b in zip(test[d], ref[d])),
                     dev_sents_set_to_dissent=dev_changed, why=why))

with open(f"{W}/SUBMISSIONS_cvs.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
with zipfile.ZipFile(f"{W}/cv_submissions_all.zip", "w") as z:
    for r in rows:
        z.write(f"{SUB}/{r['file']}", arcname=r["file"])
    z.write(f"{W}/SUBMISSIONS_cvs.csv", arcname="SUBMISSIONS_cvs.csv")
print(f"\n=== SUBMISSIONS ({len(rows)}) ===  dev max 0.846 | Dis = test Dissent sentences (docs) | "
      "vs0.6352 = test sentences different from the best file | devDis = dev sentences wrongly set to Dissent")
print(f"{'file':38s} {'dev':>6} {'mem':>4} {'vot':>4} {'k':>2} {'Dis':>9} {'vs0.6352':>8} {'devDis':>6}  why")
for r in rows:
    print(f"{r['file']:38s} {r['dev_macro_f1']:6.4f} {r['members']:4d} {r['voters']:4d} {r['k']:2d} "
          f"{r['test_dissent']:4d} ({r['test_docs_with_dissent']:2d}) {r['vs_best']:8d} {r['dev_sents_set_to_dissent']:6d}  {r['why']}")